In [ ]:
import numpy as np
import nibabel as nib
from skimage.measure import marching_cubes
from scipy.ndimage import binary_closing
from scipy.ndimage import generate_binary_structure
from nibabel.affines import apply_affine


# ============================================================
# SETTINGS
# ============================================================

LABEL_FILE = "Brain_label.nii"

OUTPUT_OBJ = "brain_segmented.obj"
OUTPUT_MTL = "brain_segmented.mtl"

# No closing initially
CLOSING_RADIUS = 0

# The paper defines labels 1-86
VALID_LABELS = range(1, 87)


# ============================================================
# COLORS FROM TABLE 1
# RGB values extracted from the uploaded image
# ============================================================

COLORS = {
    1:  (237, 216, 166),
    2:  (213, 91, 43),
    3:  (213, 91, 43),
    4:  (244, 201, 68),
    5:  (244, 201, 68),
    6:  (178, 210, 67),
    7:  (178, 210, 67),
    8:  (108, 186, 57),
    9:  (108, 186, 57),
    10: (186, 235, 189),
    11: (186, 235, 189),
    12: (83, 149, 186),
    13: (83, 149, 186),
    14: (56, 70, 132),
    15: (56, 70, 132),
    16: (172, 217, 111),
    17: (172, 217, 111),
    18: (209, 134, 193),
    19: (209, 134, 193),
    20: (203, 74, 150),
    21: (203, 74, 150),
    22: (174, 117, 72),
    23: (174, 117, 72),
    24: (243, 228, 80),
    25: (243, 228, 80),
    26: (241, 250, 215),
    27: (241, 250, 215),
    28: (126, 199, 89),
    29: (126, 199, 89),
    30: (187, 150, 147),
    31: (187, 150, 147),
    32: (248, 213, 200),
    33: (248, 213, 200),
    34: (201, 60, 45),
    35: (201, 60, 45),
    36: (140, 196, 59),
    37: (140, 196, 59),
    38: (204, 58, 50),
    39: (204, 58, 50),
    40: (192, 235, 230),
    41: (192, 235, 230),
    42: (168, 221, 229),
    43: (168, 221, 229),
    44: (60, 58, 153),
    45: (60, 58, 153),
    46: (113, 75, 174),
    47: (113, 75, 174),
    48: (245, 220, 74),
    49: (245, 220, 74),

    50: (123, 191, 57),
    51: (123, 191, 57),
    52: (233, 134, 173),
    53: (233, 134, 173),
    54: (229, 128, 61),
    55: (229, 128, 61),
    56: (246, 224, 100),
    57: (246, 224, 100),
    58: (90, 132, 53),
    59: (90, 132, 53),
    60: (191, 220, 96),
    61: (191, 220, 96),
    62: (237, 242, 251),
    63: (237, 242, 251),
    64: (203, 226, 237),
    65: (203, 226, 237),
    66: (146, 133, 204),
    67: (146, 133, 204),
    68: (185, 231, 228),
    69: (185, 231, 228),
    70: (249, 237, 139),
    71: (247, 205, 84),
    72: (148, 65, 176),
    73: (251, 245, 250),
    74: (119, 38, 29),
    75: (150, 198, 65),
    76: (219, 95, 74),
    77: (255, 255, 255),
    78: (213, 115, 44),
    79: (213, 115, 44),
    80: (176, 213, 233),
    81: (134, 88, 185),
    82: (134, 88, 185),
    83: (206, 50, 70),
    84: (206, 50, 70),
    85: (134, 88, 185),
    86: (134, 88, 185),
}


# ============================================================
# LOAD LABEL VOLUME
# ============================================================

print("Loading label volume...")

nii = nib.load(LABEL_FILE)

labels = np.asarray(nii.dataobj)
affine = nii.affine

print("Volume shape:", labels.shape)

unique_labels, counts = np.unique(labels, return_counts=True)

print("\nLabels found in NIfTI:")
for label, count in zip(unique_labels, counts):
    print(f"  Label {label}: {count} voxels")


# ============================================================
# GENERATE SURFACES
# ============================================================

all_vertices = []
all_normals = []
all_faces = []
all_face_labels = []

vertex_offset = 0

print("\nGenerating meshes...")

for label in VALID_LABELS:

    mask = labels == label

    voxel_count = np.sum(mask)

    # Skip labels that don't exist
    if voxel_count == 0:
        print(f"Label {label}: not present")
        continue

    print(f"Label {label}: {voxel_count} voxels")

    # først binær maske af hele volumen

    # --------------------------------------------------------
    # Optional morphological closing
    # --------------------------------------------------------

    if CLOSING_RADIUS > 0:

        structure = generate_binary_structure(3, 1)

        mask = binary_closing(
            mask,
            structure=structure,
            iterations=CLOSING_RADIUS
        )

    # --------------------------------------------------------
    # Marching cubes
    # --------------------------------------------------------

    try:
        vertices_voxel, faces, normals_voxel, values = marching_cubes(
        mask.astype(np.float32),
        level=0.5
    )

    # også gem normals til shading 

    except ValueError:

        print(f"  -> Could not generate surface for label {label}")
        continue

    # --------------------------------------------------------
    # Convert voxel coordinates -> world coordinates
    # --------------------------------------------------------

    vertices_world = apply_affine(
        affine,
        vertices_voxel
    )


    # Linear part of the NIfTI affine
    A = affine[:3, :3]

    # Normals transform using inverse-transpose
    normal_transform = np.linalg.inv(A).T

    normals_world = normals_voxel @ normal_transform

    # Normalize normals
    normals_world /= np.linalg.norm(
        normals_world,
        axis=1,
        keepdims=True
    )

    # --------------------------------------------------------
    # Store
    # --------------------------------------------------------

    all_vertices.append(vertices_world)
    all_normals.append(normals_world)

    all_faces.append(
        faces + vertex_offset
    )

    # One label value for every face
    all_face_labels.append(
        np.full(len(faces), label, dtype=np.int32)
    )

    vertex_offset += len(vertices_world)

    print(
        f"  -> {len(vertices_world):,} vertices, "
        f"{len(faces):,} faces"
    )


# ============================================================
# COMBINE ALL REGIONS
# ============================================================

vertices = np.vstack(all_vertices)
normals = np.vstack(all_normals)
faces = np.vstack(all_faces)
face_labels = np.concatenate(all_face_labels)

print("\n===================================")
print("Final mesh")
print("===================================")

print("Vertices:", len(vertices))
print("Faces:", len(faces))
print("Labels:", np.unique(face_labels))


# ============================================================
# WRITE MTL FILE
# ============================================================

print("\nWriting MTL file...")

with open(OUTPUT_MTL, "w") as f:

    f.write("# Brain anatomical region materials\n\n")

    for label in sorted(np.unique(face_labels)):

        r, g, b = COLORS[int(label)]

        # OBJ/MTL uses values from 0 to 1
        r /= 255
        g /= 255
        b /= 255

        f.write(f"newmtl label_{label}\n")

        # Diffuse color
        f.write(f"Kd {r:.6f} {g:.6f} {b:.6f}\n")

        # Ambient color
        f.write(f"Ka {r:.6f} {g:.6f} {b:.6f}\n")

        # Specular color
        f.write("Ks 0.000000 0.000000 0.000000\n")

        # No transparency
        f.write("d 1.0\n")

        f.write("\n")


# ============================================================
# WRITE OBJ FILE
# ============================================================

print("Writing OBJ file...")

with open(OUTPUT_OBJ, "w") as f:

    f.write("# Brain anatomical region mesh\n")
    f.write(f"mtllib {OUTPUT_MTL}\n\n")

    # --------------------------------------------------------
    # Vertices
    # --------------------------------------------------------

    for x, y, z in vertices:

        f.write(
            f"v {x:.6f} {y:.6f} {z:.6f}\n"
        )

    f.write("\n")

    # --------------------------------------------------------
    # Vertex normals
    # --------------------------------------------------------

    for nx, ny, nz in normals:

        f.write(
            f"vn {nx:.6f} {ny:.6f} {nz:.6f}\n"
        )

    f.write("\n")

    # --------------------------------------------------------
    # Faces grouped by anatomical label
    # --------------------------------------------------------

    for label in sorted(np.unique(face_labels)):

        f.write(f"usemtl label_{label}\n")

        label_faces = faces[face_labels == label]

        for face in label_faces:

            # OBJ uses 1-based indexing
            v1 = face[0] + 1
            v2 = face[1] + 1
            v3 = face[2] + 1

            # vertex//normal
            f.write(
                f"f {v1}//{v1} {v2}//{v2} {v3}//{v3}\n"
            )

    f.write("\n")


print("\n===================================")
print("DONE")
print("===================================")

print(f"OBJ: {OUTPUT_OBJ}")
print(f"MTL: {OUTPUT_MTL}")
print(f"Labels included: {len(np.unique(face_labels))}")

Loading label volume...
Volume shape: (149, 208, 140)

Labels found in NIfTI:
  Label 0: 3524400 voxels
  Label 1: 72390 voxels
  Label 2: 3180 voxels
  Label 3: 3986 voxels
  Label 4: 16048 voxels
  Label 5: 15654 voxels
  Label 6: 12099 voxels
  Label 7: 11067 voxels
  Label 8: 9696 voxels
  Label 9: 9986 voxels
  Label 10: 15019 voxels
  Label 11: 14161 voxels
  Label 12: 13381 voxels
  Label 13: 15163 voxels
  Label 14: 14938 voxels
  Label 15: 14064 voxels
  Label 16: 7242 voxels
  Label 17: 9884 voxels
  Label 18: 12030 voxels
  Label 19: 13073 voxels
  Label 20: 12498 voxels
  Label 21: 13667 voxels
  Label 22: 12575 voxels
  Label 23: 11019 voxels
  Label 24: 7425 voxels
  Label 25: 9011 voxels
  Label 26: 8817 voxels
  Label 27: 9511 voxels
  Label 28: 7267 voxels
  Label 29: 6393 voxels
  Label 30: 5087 voxels
  Label 31: 6376 voxels
  Label 32: 6405 voxels
  Label 33: 5309 voxels
  Label 34: 9526 voxels
  Label 35: 8687 voxels
  Label 36: 3922 voxels
  Label 37: 3404 voxels
